# 13 - External Execution Quality Benchmark (Rule 605 context)

## Scientific objective
Define a reproducible protocol to benchmark internal execution-quality metrics against public Rule 605 style disclosures.

Important:
- This is a benchmark layer, not a 1:1 replication.
- Use consistent methodology and caveats by venue/order-type differences.

Reference:
- SEC Rule 605 update (2024): https://www.sec.gov/newsroom/press-releases/2024-32


## Step 1 - Setup


In [1]:
NOTEBOOK_ID = "13_execution_quality_external_benchmark_605"

import json
import os
import sys
import uuid
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

try:
    import matplotlib.pyplot as plt
except Exception as e:
    plt = None
    print("matplotlib not available:", e)

def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for cand in candidates:
        if (cand / "data" / "manifests").exists() and (cand / "notebooks" / "01_data_integrity").exists():
            return cand
    return cwd

PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DATA_ROOT = Path(os.getenv("DATA_CACHE_DIR", r"C:\TSIS_Data\data")).resolve()
SYMBOLS = ["AABA"]
MAX_FILES = 300
ROWS_PER_FILE = 2000

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUT_DIR = PROJECT_ROOT / "runs" / "data_quality" / NOTEBOOK_ID / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT).decode().strip()
except Exception:
    git_commit = "<unknown>"

print("Run ID:", RUN_ID)
print("Out dir:", OUT_DIR)
print("Data root:", DATA_ROOT)
print("Symbols:", SYMBOLS)


Run ID: 20260211_191308_1e72ab0e
Out dir: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\11_external_execution_benchmark_605\20260211_191308_1e72ab0e
Data root: C:\TSIS_Data\data
Symbols: ['AABA']


## Step 2 - Load internal metrics and external benchmark placeholders
What we do:
- Try loading internal execution metrics (if available).
- Load external benchmark table (manual csv/parquet input).

This notebook remains valid even if one side is missing; it will export a structured gap report.


In [2]:
INTERNAL_METRICS_PATH = PROJECT_ROOT / "runs" / "research" / "execution_quality" / "internal_execution_metrics.parquet"
EXTERNAL_BENCHMARK_PATH = PROJECT_ROOT / "data" / "reference" / "rule605_benchmark.csv"

internal = pl.read_parquet(INTERNAL_METRICS_PATH) if INTERNAL_METRICS_PATH.exists() else pl.DataFrame()
external = pl.read_csv(EXTERNAL_BENCHMARK_PATH) if EXTERNAL_BENCHMARK_PATH.exists() else pl.DataFrame()

print("Internal metrics rows:", internal.height)
print("External benchmark rows:", external.height)

if internal.height > 0 and external.height > 0:
    join_cols = [c for c in ["symbol", "session", "bucket"] if c in internal.columns and c in external.columns]
    if join_cols:
        comp = internal.join(external, on=join_cols, how="inner", suffix="_ext")
    else:
        comp = pl.DataFrame()
else:
    comp = pl.DataFrame()

comp.head(10) if comp.height > 0 else comp


Internal metrics rows: 0
External benchmark rows: 0


shape: (0, 0)
┌┐
╞╡
└┘

## Step 3 - Gap report and export


In [3]:
comparison_rows = int(comp.height)
applicability_status = "APPLICABLE" if (internal.height > 0 and external.height > 0) else "NOT_APPLICABLE"
overall_status = "PASS" if comparison_rows > 0 else ("NOT_APPLICABLE" if applicability_status == "NOT_APPLICABLE" else "WARN")

if applicability_status == "NOT_APPLICABLE":
    root_cause = "schema_gap"
elif overall_status in {"WARN", "FAIL"}:
    root_cause = "sample_issue"
else:
    root_cause = "none"

gate_breakdown = [
    {"gate": "internal_source_available", "status": "PASS" if internal.height > 0 else "FAIL", "value": int(internal.height)},
    {"gate": "external_benchmark_available", "status": "PASS" if external.height > 0 else "FAIL", "value": int(external.height)},
    {"gate": "join_rows_positive", "status": "PASS" if comparison_rows > 0 else "WARN", "value": comparison_rows},
]

decision_table = [{
    "ticker": SYMBOLS[0] if SYMBOLS else "<unknown>",
    "applicability_status": applicability_status,
    "overall_status": overall_status,
    "root_cause": root_cause,
    "comparison_rows": comparison_rows,
    "decision": overall_status,
}]

gap_report = {
    "check_id": "execution_quality_external_benchmark_605_v1",
    "as_of_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "symbols": SYMBOLS,
    "internal_available": internal.height > 0,
    "external_available": external.height > 0,
    "comparison_rows": comparison_rows,
    "applicability_status": applicability_status,
    "overall_status": overall_status,
    "root_cause": root_cause,
    "gate_breakdown": gate_breakdown,
    "decision_table": decision_table,
    "next_action": "populate internal/external benchmark sources if comparison_rows == 0",
}

if internal.height > 0:
    internal.write_parquet(OUT_DIR / "internal_metrics_snapshot.parquet")
if external.height > 0:
    external.write_parquet(OUT_DIR / "external_benchmark_snapshot.parquet")
if comp.height > 0:
    comp.write_parquet(OUT_DIR / "execution_benchmark_comparison.parquet")

with open(OUT_DIR / "execution_benchmark_gap_report.json", "w", encoding="utf-8") as f:
    json.dump(gap_report, f, indent=2)

print("Saved:", OUT_DIR)
print(gap_report)


Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\11_external_execution_benchmark_605\20260211_191308_1e72ab0e
{'internal_available': False, 'external_available': False, 'comparison_rows': 0, 'next_action': 'populate internal/external benchmark sources if comparison_rows == 0'}
